# 06 - Gold marts (business-ready, one revenue definition)

**Inputs:** accepted `silver_orders` and `silver_transactions` only (never Bronze or quarantine).

**Outputs**
| Table | Grain | Unique key | Purpose |
|---|---|---|---|
| `gold_daily_sales` | one row per `order_date` | `order_date` | daily sales and cancellation rate |
| `gold_revenue_breakdown` | one row per category + country | `category, country` | revenue by category and country |
| `gold_payment_success` | one row per `payment_method` | `payment_method` | payment success rate by method |
| `gold_top_customers` | one row per customer | `customer_id` | top customers by spend |

### The pattern that prevents double counting
1. Aggregate eligible payments to **one row per order** (`order_payments`).
2. Join that to **one row per order** (`order_revenue`). Row count must equal `silver_orders`.
3. Every mart is built from `order_revenue` (or from converted transactions for payment success).

Transactions are never joined directly to orders in a mart, so an order with 3 payments cannot be counted 3 times.

### Important
The revenue policy is a **business decision you document**, not an assignment-provided rule.
Statuses and rates below are placeholders until you check them against **your** Silver data (section 3).
Run `05_reconciliation` first so Gold is built on reconciled Silver.

## 1. Parameters (these ARE your revenue policy)

The widgets below define the revenue policy. Section 3 verifies these defaults against the actual Silver data and blocks the run if they don't match.

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")

# --- Which orders/payments count (PLACEHOLDER defaults - verify in section 3) ---
dbutils.widgets.text("eligible_order_statuses", "shipped", "Order statuses that qualify for revenue")
dbutils.widgets.text("cancelled_order_statuses", "cancelled", "Order statuses counted as cancelled")
dbutils.widgets.text("successful_txn_statuses", "success", "Txn statuses that count as a successful payment")
dbutils.widgets.text("refund_txn_statuses", "refunded", "Txn statuses that are refunds")

# --- Currency policy ---
dbutils.widgets.text("reporting_currency", "USD", "Reporting currency (UPPERCASE)")
dbutils.widgets.text("fx_rates", "EUR=1.08,GBP=1.27", "Static rates to reporting currency, e.g. EUR=1.08,GBP=1.27 (blank = none)")

# --- Top customers ---
dbutils.widgets.text("top_n", "100", "Top N customers (ties at the cut-off are kept)")

import datetime
from decimal import Decimal

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()


def parse_list(text, transform):
    return [transform(x.strip()) for x in text.split(",") if x.strip()]


def sql_in_list(values):
    """['a','b'] -> 'a', 'b' (quoted for a SQL IN (...) list, quotes escaped)"""
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


def in_list(col, values):
    """col IN (...) ; an empty list means 'nothing qualifies' -> FALSE (an empty IN () is invalid SQL)."""
    return f"{col} IN ({sql_in_list(values)})" if values else "FALSE"


def esc(text):
    """Escape a string for use inside a SQL single-quoted literal."""
    return text.replace("'", "''")


ELIGIBLE_ORDER_STATUSES = parse_list(dbutils.widgets.get("eligible_order_statuses"), str.lower)
CANCELLED_ORDER_STATUSES = parse_list(dbutils.widgets.get("cancelled_order_statuses"), str.lower)
SUCCESS_TXN_STATUSES = parse_list(dbutils.widgets.get("successful_txn_statuses"), str.lower)
REFUND_TXN_STATUSES = parse_list(dbutils.widgets.get("refund_txn_statuses"), str.lower)

REPORTING = dbutils.widgets.get("reporting_currency").strip().upper()
TOP_N = int(dbutils.widgets.get("top_n"))

# FX rates: "EUR=1.08, GBP=1.27" -> {"EUR": Decimal("1.08"), ...}. Meaning: 1 unit of EUR = 1.08 reporting-currency units.
FX = {REPORTING: Decimal("1")}
for item in [x for x in dbutils.widgets.get("fx_rates").split(",") if x.strip()]:
    ccy, rate = item.split("=")
    ccy, rate = ccy.strip().upper(), Decimal(rate.strip())
    assert rate > 0, f"FX rate for {ccy} must be positive."
    assert ccy != REPORTING, f"Do not set a rate for the reporting currency {REPORTING}; it is always 1."
    FX[ccy] = rate

# Policy sanity checks
assert ELIGIBLE_ORDER_STATUSES, "eligible_order_statuses cannot be empty."
assert SUCCESS_TXN_STATUSES, "successful_txn_statuses cannot be empty."
assert not (set(ELIGIBLE_ORDER_STATUSES) & set(CANCELLED_ORDER_STATUSES)), "An order status cannot be both eligible and cancelled."
assert not (set(SUCCESS_TXN_STATUSES) & set(REFUND_TXN_STATUSES)), "A txn status cannot be both successful and a refund."
assert TOP_N > 0, "top_n must be positive."

# Inline rates table, usable in any SQL (also in the SQL editor, so sql/03_gold_validation.sql is self-contained)
FX_TABLE_SQL = "(VALUES " + ", ".join(
    f"('{c}', CAST({r} AS DECIMAL(18,6)))" for c, r in sorted(FX.items())
) + ") AS fx(currency, rate)"

# Object names
def fq(name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

SILVER_ORDERS = fq("silver_orders")
SILVER_TXN = fq("silver_transactions")
GOLD_DAILY = fq("gold_daily_sales")
GOLD_BREAKDOWN = fq("gold_revenue_breakdown")
GOLD_PAYMENT = fq("gold_payment_success")
GOLD_CUSTOMERS = fq("gold_top_customers")

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

RUN_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()

print("Eligible order statuses :", ELIGIBLE_ORDER_STATUSES)
print("Cancelled order statuses:", CANCELLED_ORDER_STATUSES or "none defined")
print("Successful txn statuses :", SUCCESS_TXN_STATUSES)
print("Refund txn statuses     :", REFUND_TXN_STATUSES or "none defined")
print("Reporting currency      :", REPORTING)
print("FX rates to reporting   :", {k: str(v) for k, v in FX.items()})
print("Top N customers         :", TOP_N)

## 2. Input checks

Verifies that both Silver source tables exist and have rows. The run stops if either is empty (run the upstream Silver notebooks first).

In [0]:
silver_orders_n = spark.table(SILVER_ORDERS).count()
silver_txn_n = spark.table(SILVER_TXN).count()

print(f"silver_orders       : {silver_orders_n:,}")
print(f"silver_transactions : {silver_txn_n:,}")
assert silver_orders_n > 0, "silver_orders is empty - run 03_silver_orders first."
assert silver_txn_n > 0, "silver_transactions is empty - run 04_silver_transactions first."

## 3. Pre-flight: does the policy match your real data?

Each status/currency is tagged with the role your policy gives it. Check that nothing important is
mislabelled `other` before building marts.

**Output:** Three tables showing order statuses, transaction statuses, and currencies — each classified by the policy role (eligible/cancelled/success/refund/no-rate).

In [0]:
display(spark.sql(f"""
    SELECT order_status,
           CASE WHEN {in_list('order_status', ELIGIBLE_ORDER_STATUSES)}  THEN 'eligible for revenue'
                WHEN {in_list('order_status', CANCELLED_ORDER_STATUSES)} THEN 'cancelled'
                ELSE 'other (not eligible, not cancelled)' END AS policy_role,
           COUNT(*) AS orders
    FROM {SILVER_ORDERS}
    GROUP BY 1, 2
    ORDER BY orders DESC
"""))

display(spark.sql(f"""
    SELECT txn_status,
           CASE WHEN {in_list('txn_status', SUCCESS_TXN_STATUSES)} THEN 'successful payment'
                WHEN {in_list('txn_status', REFUND_TXN_STATUSES)}  THEN 'refund'
                ELSE 'other (not successful, not refund)' END AS policy_role,
           COUNT(*) AS transactions
    FROM {SILVER_TXN}
    GROUP BY 1, 2
    ORDER BY transactions DESC
"""))

display(spark.sql(f"""
    SELECT t.currency,
           COUNT(*) AS transactions,
           CASE WHEN fx.currency IS NULL THEN 'NO RATE - add to fx_rates or exclude' ELSE 'rate available' END AS conversion
    FROM {SILVER_TXN} t
    LEFT JOIN {FX_TABLE_SQL} ON t.currency = fx.currency
    GROUP BY t.currency, fx.currency
    ORDER BY transactions DESC
"""))

### Policy gate

The run stops here if:
- a configured status never occurs in Silver (revenue would silently be zero or wrong), or
- a currency in Silver has no conversion rate. **Different currencies are never added together without a documented rate.**

**Output:** `POLICY GATE: PASSED` or `POLICY GATE: FAILED` with a list of problems to fix.

In [0]:
present_order_statuses = {r["order_status"] for r in spark.sql(f"SELECT DISTINCT order_status FROM {SILVER_ORDERS}").collect()}
present_txn_statuses = {r["txn_status"] for r in spark.sql(f"SELECT DISTINCT txn_status FROM {SILVER_TXN}").collect()}
present_currencies = {r["currency"] for r in spark.sql(f"SELECT DISTINCT currency FROM {SILVER_TXN}").collect()}

problems = []

for label, configured, present in [
    ("eligible_order_statuses", ELIGIBLE_ORDER_STATUSES, present_order_statuses),
    ("cancelled_order_statuses", CANCELLED_ORDER_STATUSES, present_order_statuses),
    ("successful_txn_statuses", SUCCESS_TXN_STATUSES, present_txn_statuses),
    ("refund_txn_statuses", REFUND_TXN_STATUSES, present_txn_statuses),
]:
    unknown = sorted(set(configured) - present)
    if unknown:
        problems.append(f"{label} contains values that do not exist in Silver: {unknown}. Present values: {sorted(present)}")

no_rate = sorted(present_currencies - set(FX))
if no_rate:
    problems.append(
        f"No conversion rate to {REPORTING} for currencies {no_rate}. "
        "Add them to the fx_rates widget (e.g. EUR=1.08) or fix the Silver currency list. "
        "Currencies are never added together without a documented rate."
    )

if problems:
    print("POLICY GATE: FAILED\n")
    for p in problems:
        print(" -", p)
else:
    print("POLICY GATE: PASSED")
    print(f"  currencies in Silver: {sorted(present_currencies)}  ->  reporting currency {REPORTING}")
    if len(present_currencies) > 1 or present_currencies != {REPORTING}:
        print("  NOTE: amounts are converted with STATIC, documented rates (assumptions, not market data).")

assert not problems, "Fix the policy widgets (see messages above) and rerun."

## 4. Revenue definition (the 8 items the guide requires)

Printed from the widget values actually used. Copy it into the README.

**Output:** A formatted markdown document defining the revenue policy — eligible statuses, refund handling, currency conversion, partial payments, and source of truth.

In [0]:
fx_text = "single currency, no conversion needed" if set(FX) == {REPORTING} else \
    "converted with static documented rates: " + ", ".join(f"1 {c} = {r} {REPORTING}" for c, r in sorted(FX.items()) if c != REPORTING)

REVENUE_POLICY_MD = f"""
## Revenue definition (used everywhere: Databricks SQL, AI/BI dashboard, Power BI)

**Eligible revenue** = money from *successful payment transactions* on *eligible orders*, attributed to the **order date**,
expressed in **{REPORTING}**.

| # | Question | Decision |
|---|---|---|
| 1 | Which order statuses qualify? | {ELIGIBLE_ORDER_STATUSES} |
| 2 | Which payment statuses qualify? | {SUCCESS_TXN_STATUSES} |
| 3 | Are cancelled orders excluded? | Yes. Statuses {CANCELLED_ORDER_STATUSES or 'none defined'} are counted as cancelled and carry no eligible revenue. Any order whose status is not in the eligible list is excluded from revenue. |
| 4 | How are refunds treated? | Transactions with status {REFUND_TXN_STATUSES or 'none defined'} are **not** successful payments, so they are never in eligible revenue. They are **not subtracted** again (that would double-count the exclusion). Their amount is reported separately as `refunded_payment_amount` (informational). |
| 5 | How are currencies handled? | {fx_text}. Amounts are converted per transaction and rounded to 2 decimals. Orders carry no currency; currency comes from the payment. |
| 6 | Do partial payments count? | Yes, whatever was actually paid by successful transactions counts. Nothing is topped up to the order value. |
| 7 | Multiple successful payments / overpayment? | All successful payments on an eligible order are summed. Overpayment is **not capped**; it is only observed (section 6). |
| 8 | Source of truth | `silver_transactions` (amounts, statuses, currency) joined at **order level** to `silver_orders` (status, date, category, country, customer). Only accepted Silver rows are used. |

**Order-level dataset:** `order_revenue` has exactly one row per accepted order. Payments are aggregated per order *before* the join, so payments cannot multiply order rows.

**Cancellation rate** = cancelled orders / all accepted orders on the same order date (same definition in every report).

**Payment success rate** = successful transactions / all accepted transactions for the payment method.
"""
print(REVENUE_POLICY_MD)

## 5. Step A - order-level revenue dataset

Temp views (nothing persisted yet):
- `txn_converted` - each accepted transaction in the reporting currency, flagged successful / refund
- `order_payments` - **one row per order** with summed payments
- `order_revenue` - **one row per accepted order** with its eligible revenue

**Output:** Row-count check confirming `order_revenue` has exactly one row per accepted order (no join fan-out).

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_converted AS
SELECT
    t.txn_id, t.order_id, t.payment_method, t.currency, t.txn_status, t.txn_ts,
    t.amount                                       AS amount_original,
    fx.rate                                        AS fx_rate,
    CAST(ROUND(t.amount * fx.rate, 2) AS DECIMAL(28,2)) AS amount_reporting,
    {in_list('t.txn_status', SUCCESS_TXN_STATUSES)}  AS is_success,
    {in_list('t.txn_status', REFUND_TXN_STATUSES)}   AS is_refund
FROM {SILVER_TXN} t
LEFT JOIN {FX_TABLE_SQL} ON t.currency = fx.currency
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW order_payments AS
SELECT
    order_id,
    CAST(COUNT(*) AS BIGINT)                                                         AS txn_count,
    CAST(SUM(CASE WHEN is_success THEN 1 ELSE 0 END) AS BIGINT)                      AS successful_txn_count,
    CAST(SUM(CASE WHEN is_success THEN amount_reporting ELSE 0 END) AS DECIMAL(28,2)) AS paid_amount,
    CAST(SUM(CASE WHEN is_refund  THEN 1 ELSE 0 END) AS BIGINT)                      AS refunded_txn_count,
    CAST(SUM(CASE WHEN is_refund  THEN amount_reporting ELSE 0 END) AS DECIMAL(28,2)) AS refunded_amount
FROM txn_converted
GROUP BY order_id
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW order_revenue AS
SELECT
    o.order_id, o.customer_id, o.order_date, o.category, o.country, o.order_status,
    o.quantity, o.unit_price,
    CAST(o.quantity * o.unit_price AS DECIMAL(28,2))                      AS order_gross_value,
    {in_list('o.order_status', ELIGIBLE_ORDER_STATUSES)}                  AS is_eligible_order,
    {in_list('o.order_status', CANCELLED_ORDER_STATUSES)}                 AS is_cancelled,
    COALESCE(p.txn_count, 0)                                              AS txn_count,
    COALESCE(p.successful_txn_count, 0)                                   AS successful_txn_count,
    COALESCE(p.paid_amount, CAST(0 AS DECIMAL(28,2)))                     AS paid_amount,
    COALESCE(p.refunded_txn_count, 0)                                     AS refunded_txn_count,
    COALESCE(p.refunded_amount, CAST(0 AS DECIMAL(28,2)))                 AS refunded_amount,
    CASE WHEN {in_list('o.order_status', ELIGIBLE_ORDER_STATUSES)}
         THEN COALESCE(p.paid_amount, CAST(0 AS DECIMAL(28,2)))
         ELSE CAST(0 AS DECIMAL(28,2)) END                                AS eligible_revenue
FROM {SILVER_ORDERS} o
LEFT JOIN order_payments p ON o.order_id = p.order_id
""")

# Fan-out guard: the order-level dataset MUST have exactly one row per accepted order
stats = spark.sql("SELECT COUNT(*) AS n, COUNT(DISTINCT order_id) AS d FROM order_revenue").collect()[0]
print(f"order_revenue rows: {stats['n']:,}   distinct order_id: {stats['d']:,}   silver_orders: {silver_orders_n:,}")
assert stats["n"] == stats["d"] == silver_orders_n, "order_revenue is not one row per order - a join is multiplying rows."
print("OK: one row per order, no join fan-out.")

## 6. What did the policy do? (informational)

These numbers show the impact of each decision so the README can state it honestly.
The over/under-payment comparison **assumes order prices are in the reporting currency** (orders have no currency field),
so treat it as an observation, not a rule.

**Output:** A table with 8 measures showing how many orders are eligible, how many have zero revenue, excluded payments, partial/overpayments, and refunds.

In [0]:
display(spark.sql("""
    SELECT 'orders (all accepted)'                                   AS measure, COUNT(*) AS orders, CAST(NULL AS DECIMAL(28,2)) AS amount FROM order_revenue
    UNION ALL
    SELECT 'eligible orders (status qualifies)',                     COUNT(*), CAST(NULL AS DECIMAL(28,2)) FROM order_revenue WHERE is_eligible_order
    UNION ALL
    SELECT 'eligible orders with NO successful payment (zero revenue)', COUNT(*), CAST(NULL AS DECIMAL(28,2)) FROM order_revenue WHERE is_eligible_order AND paid_amount = 0
    UNION ALL
    SELECT 'NON-eligible orders that HAVE successful payments (excluded money)', COUNT(*), SUM(paid_amount) FROM order_revenue WHERE NOT is_eligible_order AND paid_amount > 0
    UNION ALL
    SELECT 'eligible orders with 2+ successful payments (summed)',   COUNT(*), SUM(paid_amount) FROM order_revenue WHERE is_eligible_order AND successful_txn_count >= 2
    UNION ALL
    SELECT 'eligible orders where payments < order value (partial, observed)', COUNT(*), SUM(order_gross_value - paid_amount) FROM order_revenue WHERE is_eligible_order AND paid_amount > 0 AND paid_amount < order_gross_value
    UNION ALL
    SELECT 'eligible orders where payments > order value (overpaid, observed, NOT capped)', COUNT(*), SUM(paid_amount - order_gross_value) FROM order_revenue WHERE is_eligible_order AND paid_amount > order_gross_value
    UNION ALL
    SELECT 'orders with refunded payments (not deducted)',           COUNT(*), SUM(refunded_amount) FROM order_revenue WHERE refunded_txn_count > 0
"""))

## 7. Step B - table documentation (descriptions, grain, key, formulas)

One dictionary drives the table comments, column comments and the README text, so they cannot disagree.

**Output:** Prints the list of tables for which documentation is defined.

In [0]:
CANCEL_DEF = "cancelled orders / all accepted orders on the same order date"

TABLE_DOCS = {
    "gold_daily_sales": {
        "description": "Daily sales and cancellation rate from accepted Silver orders and successful payments.",
        "grain": "one row per order_date",
        "key": "order_date",
        "metrics": {
            "order_count": "count of all accepted orders on the order date",
            "eligible_order_count": f"orders whose status is in {ELIGIBLE_ORDER_STATUSES}",
            "paid_order_count": "eligible orders with at least some successful payment",
            "cancelled_order_count": f"orders whose status is in {CANCELLED_ORDER_STATUSES}",
            "cancellation_rate": CANCEL_DEF,
            "eligible_revenue": f"sum of successful payments on eligible orders, in {REPORTING}, attributed to order date",
            "refunded_payment_amount": "informational: refunded payments on that date's orders; NOT deducted from revenue",
            "avg_revenue_per_paid_order": "eligible_revenue / paid_order_count",
        },
        "columns": {
            "order_date": "Order date (grain; unique key)",
            "reporting_currency": "Currency of all monetary columns",
            "order_count": "All accepted orders on this date. Denominator of cancellation_rate",
            "eligible_order_count": "Orders with a revenue-eligible status",
            "paid_order_count": "Eligible orders with at least one successful payment",
            "cancelled_order_count": "Orders with a cancelled status",
            "cancellation_rate": "cancelled_order_count / order_count (0 to 1)",
            "eligible_revenue": "Successful payments on eligible orders in reporting currency",
            "refunded_payment_amount": "Informational refunded payment amount; not deducted from eligible_revenue",
            "avg_revenue_per_paid_order": "eligible_revenue / paid_order_count (NULL when no paid orders)",
            "_gold_processed_at": "When this Gold row was built",
        },
    },
    "gold_revenue_breakdown": {
        "description": "Revenue by product category and country from accepted Silver orders and successful payments.",
        "grain": "one row per category and country",
        "key": "category, country",
        "metrics": {
            "order_count": "all accepted orders in the category/country",
            "eligible_order_count": f"orders whose status is in {ELIGIBLE_ORDER_STATUSES}",
            "paid_order_count": "eligible orders with at least some successful payment",
            "customer_count": "distinct customers with any accepted order",
            "paying_customer_count": "distinct customers with at least one paid eligible order",
            "eligible_revenue": f"sum of successful payments on eligible orders, in {REPORTING}",
            "revenue_share": "eligible_revenue / total eligible_revenue across all rows (0 to 1)",
        },
        "columns": {
            "category": "Normalized product category (part of key)",
            "country": "Normalized country (part of key)",
            "reporting_currency": "Currency of all monetary columns",
            "order_count": "All accepted orders in this category/country",
            "eligible_order_count": "Orders with a revenue-eligible status",
            "paid_order_count": "Eligible orders with at least one successful payment",
            "customer_count": "Distinct customers with any accepted order here. Not additive across rows",
            "paying_customer_count": "Distinct customers with a paid eligible order here. Not additive across rows",
            "eligible_revenue": "Successful payments on eligible orders in reporting currency",
            "revenue_share": "Share of total eligible revenue (0 to 1)",
            "_gold_processed_at": "When this Gold row was built",
        },
    },
    "gold_payment_success": {
        "description": "Payment performance by payment method from accepted Silver transactions (all statuses).",
        "grain": "one row per payment_method",
        "key": "payment_method",
        "metrics": {
            "transaction_count": "all accepted transactions for the method",
            "successful_transaction_count": f"transactions whose status is in {SUCCESS_TXN_STATUSES}",
            "success_rate": "successful_transaction_count / transaction_count",
            "refunded_transaction_count": f"transactions whose status is in {REFUND_TXN_STATUSES}",
            "other_transaction_count": "transactions that are neither successful nor refunded",
            "total_amount": f"sum of all transaction amounts in {REPORTING}",
            "successful_amount": f"sum of successful transaction amounts in {REPORTING}",
        },
        "columns": {
            "payment_method": "Normalized payment method (grain; unique key)",
            "reporting_currency": "Currency of all monetary columns",
            "transaction_count": "All accepted transactions for this method. Denominator of success_rate",
            "successful_transaction_count": "Transactions with a successful status",
            "refunded_transaction_count": "Transactions with a refund status",
            "other_transaction_count": "Transactions that are neither successful nor refunded",
            "success_rate": "successful_transaction_count / transaction_count (0 to 1)",
            "total_amount": "Sum of all transaction amounts in reporting currency",
            "successful_amount": "Sum of successful transaction amounts in reporting currency",
            "_gold_processed_at": "When this Gold row was built",
        },
    },
    "gold_top_customers": {
        "description": f"Top {TOP_N} customers by eligible spend (ties at the cut-off are kept).",
        "grain": "one row per customer_id",
        "key": "customer_id",
        "metrics": {
            "eligible_spend": f"sum of eligible revenue over the customer's orders, in {REPORTING}",
            "eligible_order_count": f"customer's orders whose status is in {ELIGIBLE_ORDER_STATUSES}",
            "paid_order_count": "eligible orders with at least some successful payment",
            "total_order_count": "all accepted orders of the customer",
            "spend_rank": "RANK() by eligible_spend DESC; equal spend shares the same rank",
            "rank_position": "ROW_NUMBER() by eligible_spend DESC, paid_order_count DESC, customer_id ASC; always unique",
        },
        "columns": {
            "customer_id": "Customer (grain; unique key)",
            "reporting_currency": "Currency of all monetary columns",
            "eligible_spend": "Eligible revenue across the customer's orders in reporting currency",
            "eligible_order_count": "Customer's orders with a revenue-eligible status",
            "paid_order_count": "Eligible orders with at least one successful payment",
            "total_order_count": "All accepted orders of the customer",
            "spend_rank": "Rank by spend; ties share a rank (RANK semantics, gaps follow ties)",
            "rank_position": "Unique position; ties broken by paid_order_count DESC then customer_id ASC",
            "_gold_processed_at": "When this Gold row was built",
        },
    },
}

print("Documentation defined for:", list(TABLE_DOCS))

## 8. Step C - build the four marts

Each table is rebuilt with `CREATE OR REPLACE TABLE`, so reruns never duplicate rows.
Table and column descriptions are attached to the tables in Unity Catalog.

**Output:** Row counts for each mart table as it is written.

In [0]:
def write_mart(table_name, fq_name, select_sql):
    """Create/replace a Gold table from a SELECT, then attach table and column comments."""
    doc = TABLE_DOCS[table_name]
    table_comment = esc(f"{doc['description']} Grain: {doc['grain']}. Unique key: {doc['key']}.")
    spark.sql(f"""
        CREATE OR REPLACE TABLE {fq_name}
        USING DELTA
        COMMENT '{table_comment}'
        AS {select_sql}
    """)
    existing = {f.name for f in spark.table(fq_name).schema.fields}
    for col, comment in doc["columns"].items():
        if col in existing:
            try:
                spark.sql(f"ALTER TABLE {fq_name} ALTER COLUMN `{col}` COMMENT '{esc(comment)}'")
            except Exception as e:
                print(f"  (note) column comment skipped for {table_name}.{col}: {type(e).__name__}")
    missing_docs = existing - set(doc["columns"])
    assert not missing_docs, f"{table_name}: columns without documentation: {missing_docs}"
    n = spark.table(fq_name).count()
    print(f"{table_name}: {n:,} rows written")
    return n

In [0]:
# A. gold_daily_sales  (grain: order_date)
daily_rows = write_mart("gold_daily_sales", GOLD_DAILY, f"""
SELECT
    order_date,
    '{REPORTING}'                                                                     AS reporting_currency,
    CAST(COUNT(*) AS BIGINT)                                                          AS order_count,
    CAST(SUM(CASE WHEN is_eligible_order THEN 1 ELSE 0 END) AS BIGINT)                AS eligible_order_count,
    CAST(SUM(CASE WHEN is_eligible_order AND paid_amount > 0 THEN 1 ELSE 0 END) AS BIGINT) AS paid_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS BIGINT)                     AS cancelled_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6))    AS cancellation_rate,
    CAST(SUM(eligible_revenue) AS DECIMAL(28,2))                                      AS eligible_revenue,
    CAST(SUM(refunded_amount) AS DECIMAL(28,2))                                       AS refunded_payment_amount,
    CAST(SUM(eligible_revenue)
         / NULLIF(SUM(CASE WHEN is_eligible_order AND paid_amount > 0 THEN 1 ELSE 0 END), 0) AS DECIMAL(28,2)) AS avg_revenue_per_paid_order,
    current_timestamp()                                                               AS _gold_processed_at
FROM order_revenue
GROUP BY order_date
""")

In [0]:
# B. gold_revenue_breakdown  (grain: category + country)
breakdown_rows = write_mart("gold_revenue_breakdown", GOLD_BREAKDOWN, f"""
WITH grp AS (
    SELECT
        category,
        country,
        CAST(COUNT(*) AS BIGINT)                                                      AS order_count,
        CAST(SUM(CASE WHEN is_eligible_order THEN 1 ELSE 0 END) AS BIGINT)            AS eligible_order_count,
        CAST(SUM(CASE WHEN is_eligible_order AND paid_amount > 0 THEN 1 ELSE 0 END) AS BIGINT) AS paid_order_count,
        CAST(COUNT(DISTINCT customer_id) AS BIGINT)                                   AS customer_count,
        CAST(COUNT(DISTINCT CASE WHEN is_eligible_order AND paid_amount > 0 THEN customer_id END) AS BIGINT) AS paying_customer_count,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2))                                  AS eligible_revenue
    FROM order_revenue
    GROUP BY category, country
)
SELECT
    category,
    country,
    '{REPORTING}'                                                                     AS reporting_currency,
    order_count, eligible_order_count, paid_order_count, customer_count, paying_customer_count,
    eligible_revenue,
    CAST(eligible_revenue / NULLIF(SUM(eligible_revenue) OVER (), 0) AS DECIMAL(9,6)) AS revenue_share,
    current_timestamp()                                                               AS _gold_processed_at
FROM grp
""")

In [0]:
# C. gold_payment_success  (grain: payment_method)
payment_rows = write_mart("gold_payment_success", GOLD_PAYMENT, f"""
SELECT
    payment_method,
    '{REPORTING}'                                                                     AS reporting_currency,
    CAST(COUNT(*) AS BIGINT)                                                          AS transaction_count,
    CAST(SUM(CASE WHEN is_success THEN 1 ELSE 0 END) AS BIGINT)                       AS successful_transaction_count,
    CAST(SUM(CASE WHEN is_refund  THEN 1 ELSE 0 END) AS BIGINT)                       AS refunded_transaction_count,
    CAST(SUM(CASE WHEN NOT is_success AND NOT is_refund THEN 1 ELSE 0 END) AS BIGINT) AS other_transaction_count,
    CAST(SUM(CASE WHEN is_success THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6))      AS success_rate,
    CAST(SUM(amount_reporting) AS DECIMAL(28,2))                                      AS total_amount,
    CAST(SUM(CASE WHEN is_success THEN amount_reporting ELSE 0 END) AS DECIMAL(28,2)) AS successful_amount,
    current_timestamp()                                                               AS _gold_processed_at
FROM txn_converted
GROUP BY payment_method
""")

In [0]:
# D. gold_top_customers  (grain: customer_id)
# Ties: spend_rank uses RANK (equal spend = equal rank). Every customer tied at the cut-off is kept, so the table can
# hold slightly more than top_n rows. rank_position is unique (ties broken by paid orders, then customer_id).
customers_rows = write_mart("gold_top_customers", GOLD_CUSTOMERS, f"""
WITH agg AS (
    SELECT
        customer_id,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2))                                   AS eligible_spend,
        CAST(SUM(CASE WHEN is_eligible_order THEN 1 ELSE 0 END) AS BIGINT)             AS eligible_order_count,
        CAST(SUM(CASE WHEN is_eligible_order AND paid_amount > 0 THEN 1 ELSE 0 END) AS BIGINT) AS paid_order_count,
        CAST(COUNT(*) AS BIGINT)                                                       AS total_order_count
    FROM order_revenue
    GROUP BY customer_id
    HAVING SUM(eligible_revenue) > 0
),
ranked AS (
    SELECT
        *,
        CAST(RANK() OVER (ORDER BY eligible_spend DESC) AS BIGINT)                                          AS spend_rank,
        CAST(ROW_NUMBER() OVER (ORDER BY eligible_spend DESC, paid_order_count DESC, customer_id ASC) AS BIGINT) AS rank_position
    FROM agg
)
SELECT
    customer_id,
    '{REPORTING}' AS reporting_currency,
    eligible_spend, eligible_order_count, paid_order_count, total_order_count,
    spend_rank, rank_position,
    current_timestamp() AS _gold_processed_at
FROM ranked
WHERE spend_rank <= {TOP_N}
""")

## 9. Validation (SQL assertions, expected `violations = 0`)

These run against the **persisted Gold and Silver tables only** (no temp views), and recompute revenue by a
**different query path** than the marts use. That makes them runnable in the SQL editor
and gives the same SQL for `sql/03_gold_validation.sql` (section 11).

**Output:** The number of assertions defined.

In [0]:
def independent_cte():
    """Revenue recomputed straight from Silver, independent of the order_revenue temp view."""
    return f"""WITH pay AS (
    SELECT t.order_id, SUM(ROUND(t.amount * fx.rate, 2)) AS amt
    FROM {SILVER_TXN} t
    JOIN {FX_TABLE_SQL} ON t.currency = fx.currency
    WHERE {in_list('t.txn_status', SUCCESS_TXN_STATUSES)}
    GROUP BY t.order_id
),
ord_rev AS (
    SELECT o.order_id, o.customer_id, o.order_date, o.category, o.country, p.amt AS rev
    FROM {SILVER_ORDERS} o
    JOIN pay p ON o.order_id = p.order_id
    WHERE {in_list('o.order_status', ELIGIBLE_ORDER_STATUSES)}
)"""


def eq(a, b):
    """1 violation unless the two scalar subqueries are numerically equal (NULL treated as 0)."""
    return f"SELECT CASE WHEN COALESCE(({a}), 0) = COALESCE(({b}), 0) THEN 0 ELSE 1 END AS violations"


ASSERTIONS = [
    # ---- keys ----
    ("gold_daily_sales", "key_unique", "One row per order_date",
     f"SELECT COUNT(*) AS violations FROM (SELECT order_date FROM {GOLD_DAILY} GROUP BY order_date HAVING COUNT(*) > 1)"),
    ("gold_daily_sales", "key_not_null", "order_date is never NULL",
     f"SELECT COUNT(*) AS violations FROM {GOLD_DAILY} WHERE order_date IS NULL"),
    ("gold_revenue_breakdown", "key_unique", "One row per category + country",
     f"SELECT COUNT(*) AS violations FROM (SELECT category, country FROM {GOLD_BREAKDOWN} GROUP BY category, country HAVING COUNT(*) > 1)"),
    ("gold_revenue_breakdown", "key_not_null", "category and country are never NULL",
     f"SELECT COUNT(*) AS violations FROM {GOLD_BREAKDOWN} WHERE category IS NULL OR country IS NULL"),
    ("gold_payment_success", "key_unique", "One row per payment_method",
     f"SELECT COUNT(*) AS violations FROM (SELECT payment_method FROM {GOLD_PAYMENT} GROUP BY payment_method HAVING COUNT(*) > 1)"),
    ("gold_payment_success", "key_not_null", "payment_method is never NULL",
     f"SELECT COUNT(*) AS violations FROM {GOLD_PAYMENT} WHERE payment_method IS NULL"),
    ("gold_top_customers", "key_unique", "One row per customer_id",
     f"SELECT COUNT(*) AS violations FROM (SELECT customer_id FROM {GOLD_CUSTOMERS} GROUP BY customer_id HAVING COUNT(*) > 1)"),
    ("gold_top_customers", "rank_position_unique", "rank_position is unique",
     f"SELECT COUNT(*) AS violations FROM (SELECT rank_position FROM {GOLD_CUSTOMERS} GROUP BY rank_position HAVING COUNT(*) > 1)"),

    # ---- counts tie back to Silver (nothing lost, nothing multiplied) ----
    ("gold_daily_sales", "order_count_equals_silver", "Sum of order_count equals accepted silver_orders",
     eq(f"SELECT SUM(order_count) FROM {GOLD_DAILY}", f"SELECT COUNT(*) FROM {SILVER_ORDERS}")),
    ("gold_revenue_breakdown", "order_count_equals_silver", "Sum of order_count equals accepted silver_orders",
     eq(f"SELECT SUM(order_count) FROM {GOLD_BREAKDOWN}", f"SELECT COUNT(*) FROM {SILVER_ORDERS}")),
    ("gold_payment_success", "txn_count_equals_silver", "Sum of transaction_count equals accepted silver_transactions",
     eq(f"SELECT SUM(transaction_count) FROM {GOLD_PAYMENT}", f"SELECT COUNT(*) FROM {SILVER_TXN}")),
    ("silver_transactions", "currency_without_rate", "Every Silver currency has a conversion rate",
     f"""SELECT COUNT(*) AS violations FROM {SILVER_TXN} t
LEFT ANTI JOIN {FX_TABLE_SQL} ON t.currency = fx.currency"""),

    # ---- revenue: one number everywhere, equal to an independent recomputation ----
    ("gold_daily_sales", "revenue_equals_independent", "Total eligible_revenue equals independent recomputation from Silver",
     f"{independent_cte()}\n" + eq(f"SELECT SUM(eligible_revenue) FROM {GOLD_DAILY}", "SELECT SUM(rev) FROM ord_rev")),
    ("gold_revenue_breakdown", "revenue_equals_independent", "Total eligible_revenue equals independent recomputation from Silver",
     f"{independent_cte()}\n" + eq(f"SELECT SUM(eligible_revenue) FROM {GOLD_BREAKDOWN}", "SELECT SUM(rev) FROM ord_rev")),
    ("gold_daily_sales", "revenue_equals_breakdown", "Daily total revenue equals breakdown total revenue",
     eq(f"SELECT SUM(eligible_revenue) FROM {GOLD_DAILY}", f"SELECT SUM(eligible_revenue) FROM {GOLD_BREAKDOWN}")),
    ("gold_daily_sales", "daily_revenue_by_date_matches_independent", "Every date's revenue equals the independent recomputation",
     f"""{independent_cte()}
SELECT COUNT(*) AS violations
FROM {GOLD_DAILY} d
FULL OUTER JOIN (SELECT order_date, SUM(rev) AS rev FROM ord_rev GROUP BY order_date) i ON d.order_date = i.order_date
WHERE COALESCE(d.eligible_revenue, 0) <> COALESCE(i.rev, 0)"""),

    # ---- value sanity ----
    ("gold_daily_sales", "rates_and_counts_in_range", "0 <= cancellation_rate <= 1; cancelled <= orders; paid <= eligible <= orders",
     f"""SELECT COUNT(*) AS violations FROM {GOLD_DAILY}
WHERE cancellation_rate < 0 OR cancellation_rate > 1
   OR cancelled_order_count > order_count
   OR eligible_order_count > order_count
   OR paid_order_count > eligible_order_count"""),
    ("gold_daily_sales", "cancellation_rate_formula", "cancellation_rate = cancelled_order_count / order_count",
     f"""SELECT COUNT(*) AS violations FROM {GOLD_DAILY}
WHERE ABS(cancellation_rate - CAST(cancelled_order_count AS DECIMAL(28,10)) / order_count) > 0.000001"""),
    ("gold_revenue_breakdown", "no_negative_revenue", "eligible_revenue is never negative",
     f"SELECT COUNT(*) AS violations FROM {GOLD_BREAKDOWN} WHERE eligible_revenue < 0"),
    ("gold_revenue_breakdown", "revenue_share_total", "revenue_share sums to 1 (within rounding) when revenue exists",
     f"""SELECT CASE WHEN COALESCE(SUM(eligible_revenue), 0) = 0 THEN 0
            WHEN ABS(SUM(revenue_share) - 1) < 0.001 THEN 0 ELSE 1 END AS violations
FROM {GOLD_BREAKDOWN}"""),
    ("gold_payment_success", "rates_and_counts_in_range", "0 <= success_rate <= 1; successful <= total; status buckets add up",
     f"""SELECT COUNT(*) AS violations FROM {GOLD_PAYMENT}
WHERE success_rate < 0 OR success_rate > 1
   OR successful_transaction_count > transaction_count
   OR successful_transaction_count + refunded_transaction_count + other_transaction_count <> transaction_count"""),
    ("gold_payment_success", "success_rate_formula", "success_rate = successful / total",
     f"""SELECT COUNT(*) AS violations FROM {GOLD_PAYMENT}
WHERE ABS(success_rate - CAST(successful_transaction_count AS DECIMAL(28,10)) / transaction_count) > 0.000001"""),

    # ---- top customers ----
    ("gold_top_customers", "spend_matches_independent", "Each customer's eligible_spend equals the independent recomputation",
     f"""{independent_cte()}
SELECT COUNT(*) AS violations
FROM {GOLD_CUSTOMERS} g
LEFT JOIN (SELECT customer_id, SUM(rev) AS rev FROM ord_rev GROUP BY customer_id) i ON g.customer_id = i.customer_id
WHERE i.rev IS NULL OR i.rev <> g.eligible_spend"""),
    ("gold_top_customers", "no_better_customer_left_out", "No customer outside the table spends more than the smallest included spend",
     f"""{independent_cte()}
SELECT COUNT(*) AS violations
FROM (SELECT customer_id, SUM(rev) AS rev FROM ord_rev GROUP BY customer_id) i
LEFT ANTI JOIN {GOLD_CUSTOMERS} g ON i.customer_id = g.customer_id
WHERE i.rev > (SELECT MIN(eligible_spend) FROM {GOLD_CUSTOMERS})"""),
    ("gold_top_customers", "rank_rules", "spend_rank equals RANK() by spend and stays within 1..top_n",
     f"""SELECT COUNT(*) AS violations
FROM (SELECT spend_rank, RANK() OVER (ORDER BY eligible_spend DESC) AS expected_rank FROM {GOLD_CUSTOMERS})
WHERE spend_rank <> expected_rank OR spend_rank < 1 OR spend_rank > {TOP_N}"""),
]

print(f"{len(ASSERTIONS)} assertions defined.")

In [0]:
results = []
for table, name, desc, sql in ASSERTIONS:
    rows = spark.sql(sql).collect()
    violations = int(sum(int(r["violations"] or 0) for r in rows))
    results.append((table, name, desc, violations, violations == 0))

results_df = spark.createDataFrame(
    results, "table_name STRING, assertion STRING, description STRING, violations LONG, passed BOOLEAN"
)
display(results_df.orderBy("table_name", "assertion"))

In [0]:
for table, name, desc, violations, passed in results:
    print(f"[{'PASS' if passed else 'FAIL'}] {table:24s} {name:42s} violations = {violations:,}")

GOLD_OK = all(p for *_, p in results)
print("\nOVERALL:", "GOLD VALID" if GOLD_OK else "GOLD CHECKS FAILED")
assert GOLD_OK, "Gold validation failed - see FAIL lines above."

## 10. Evidence for `evidence/05_gold/`

Row counts, descriptions, schemas and sample output for each mart.

**Output:** Row-count table, the headline revenue number, sample rows from each mart, and `DESCRIBE` output showing the table/column comments stored in Unity Catalog.

In [0]:
mart_counts = [
    ("gold_daily_sales", "order_date", daily_rows),
    ("gold_revenue_breakdown", "category, country", breakdown_rows),
    ("gold_payment_success", "payment_method", payment_rows),
    ("gold_top_customers", "customer_id", customers_rows),
]
display(spark.createDataFrame(
    [(n, TABLE_DOCS[n]["grain"], k, c) for n, k, c in mart_counts],
    "table_name STRING, grain STRING, unique_key STRING, row_count LONG"
))

display(spark.sql(f"""
    SELECT '{REPORTING}' AS reporting_currency,
           SUM(eligible_revenue)      AS total_eligible_revenue,
           SUM(order_count)           AS total_orders,
           SUM(cancelled_order_count) AS total_cancelled_orders,
           CAST(SUM(cancelled_order_count) / SUM(order_count) AS DECIMAL(9,6)) AS overall_cancellation_rate
    FROM {GOLD_DAILY}
"""))

In [0]:
display(spark.table(GOLD_DAILY).orderBy("order_date", ascending=False).limit(15))

display(spark.table(GOLD_BREAKDOWN).orderBy("eligible_revenue", ascending=False).limit(15))

display(spark.table(GOLD_PAYMENT).orderBy("transaction_count", ascending=False))

display(spark.table(GOLD_CUSTOMERS).orderBy("rank_position").limit(15))

In [0]:
display(spark.sql(f"DESCRIBE TABLE EXTENDED {GOLD_DAILY}"))

display(spark.sql(f"DESCRIBE TABLE {GOLD_BREAKDOWN}"))

display(spark.sql(f"DESCRIBE TABLE {GOLD_PAYMENT}"))

display(spark.sql(f"DESCRIBE TABLE {GOLD_CUSTOMERS}"))

## 11. SQL for `sql/03_gold_validation.sql`

Copy the printed text into that file. Each statement returns `violations`; expected value is 0.
It only uses persisted tables plus an inline rates table, so it runs in the SQL editor.

**Output:** A complete SQL validation script (printable) that can be saved to `sql/03_gold_validation.sql`.

In [0]:
sql_lines = [
    "-- sql/03_gold_validation.sql",
    f"-- Generated by notebooks/06_gold_marts.py on {RUN_AT}",
    "-- Every statement returns `violations`; the expected value is 0 (statements with UNION ALL: every row must be 0).",
    f"-- Reporting currency: {REPORTING}. Static conversion rates are inlined in each statement that needs them.",
    "",
]
for table, name, desc, sql in ASSERTIONS:
    sql_lines.append(f"-- [{table}] {name}: {desc}")
    sql_lines.append(sql.strip() + ";")
    sql_lines.append("")

sql_lines += [
    "-- Row counts",
    f"SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS row_count FROM {GOLD_DAILY}",
    f"UNION ALL SELECT 'gold_revenue_breakdown', COUNT(*) FROM {GOLD_BREAKDOWN}",
    f"UNION ALL SELECT 'gold_payment_success', COUNT(*) FROM {GOLD_PAYMENT}",
    f"UNION ALL SELECT 'gold_top_customers', COUNT(*) FROM {GOLD_CUSTOMERS};",
    "",
    "-- Headline metric that Databricks SQL, the AI/BI dashboard and Power BI must all reproduce",
    f"SELECT SUM(eligible_revenue) AS total_eligible_revenue FROM {GOLD_DAILY};",
]
print("\n".join(sql_lines))

## 12. README text: Gold documentation

Revenue policy first, then each mart's grain, key, formulas and assumptions, using this run's real row counts.

**Output:** Printable markdown text for the README, with the revenue policy, table descriptions, metric formulas, and build/validation metadata.

In [0]:
print(REVENUE_POLICY_MD)

counts_by_table = {n: c for n, _, c in mart_counts}
for name, doc in TABLE_DOCS.items():
    print(f"\n### `{name}`")
    print(f"- **Description:** {doc['description']}")
    print(f"- **Grain:** {doc['grain']}")
    print(f"- **Unique key:** {doc['key']}")
    print(f"- **Row count (this run):** {counts_by_table[name]:,}")
    print(f"- **Currency / status assumptions:** monetary columns in {REPORTING}; "
          f"eligible orders {ELIGIBLE_ORDER_STATUSES}; successful payments {SUCCESS_TXN_STATUSES}; "
          f"refunds {REFUND_TXN_STATUSES or 'none defined'}; cancelled {CANCELLED_ORDER_STATUSES or 'none defined'}")
    print("- **Metric formulas:**")
    for metric, formula in doc["metrics"].items():
        print(f"  - `{metric}` = {formula}")

print(f"\nBuilt at (UTC): {RUN_AT}")
print(f"Validation: {'ALL ' + str(len(results)) + ' assertions passed' if GOLD_OK else 'FAILED'}")

## Done

Save for `evidence/05_gold/`: the row-count table, the PASS summary, the headline revenue number, samples and `DESCRIBE` output.

**Next:** the AI/BI dashboard (built only on these four Gold tables), then Genie, Power BI and the Lakeflow Job.
The headline `total_eligible_revenue` above is the number all three reporting layers must match.